# PHYS 405 Computerized Data Analysis · Week 1 · Program and data structures
Instructor: Prof. Dr. Bora Işıldak · Department of Physics, Yeditepe University

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/Phys_405_2026_2027_Fall/blob/main/Labs/Labs1/Lab1.ipynb)

**File → Save a copy in Drive, then work in your copy.** Your work lives in your Drive copy, not in the link.

How the lab works (eight problems, about 14 minutes each, in pairs):
1. Read the problem. Write your prediction in the **Predict** cell.
2. Write or run the code, then **Compare** with your prediction.
3. Run the checks.
4. Do the **Your turn** task in the empty cell.

Today's physics: a ball with air drag, a projectile, a pendulum, a simulation clock, a pendulum data file, g with an error bar, a gamma spectrum and a small peak.
The lab uses new systems and new data: the lecture's answers do not carry over, so make your predictions fresh.

In [ ]:
import sys, math, numpy as np, matplotlib.pyplot as plt
%config InlineBackend.figure_format = 'retina'   # sharp figures on high-resolution screens
print("Python", sys.version.split()[0], "· NumPy", np.__version__)

# the two data files of today (the lecture's photogate file and the lab's pendulum file)
open("freefall.csv", "w").write("""# Free-fall photogate, ball released from rest. Height of the drop in cm.
# A missed trigger is written as 0.0000.
height_cm,trial,time_s
20,1,0.2023
20,2,0.2020
20,3,0.2025
40,1,0.2853
40,2,0.2861
40,3,0.2856
60,1,0.3501
60,2,0.0000
60,3,0.3497
80,1,0.4042
80,2,0.4034
80,3,0.4045
100,1,0.4521
100,2,0.4511
100,3,0.4526
""")
open("pendulum.csv", "w").write("""# Pendulum lab, group 3. Length from pivot to centre of bob.
# Period = (time for 10 swings) / 10.
length_m,trial,period_s
0.20,1,0.888
0.20,2,0.900
0.20,3,0.874
0.20,4,0.914
0.40,1,1.276
0.40,2,1265
0.40,3,1.265
0.40,4,1.272
0.60,1,1.551
0.60,2,1.551
0.60,3,1.563
0.60,4,1.560
0.80,1,1.794
0.80,2,1.793
0.80,3,-999
0.80,4,1.787
1.00,1,2.001
1.00,2,2.013
1.00,3,2.005
1.00,4,1.990
""")
print(open("pendulum.csv").read()[:160])

## Problem 1 · A ball falls for 10 s through air. How fast is it going?
A ball of mass 0.16 kg falls from rest with quadratic air drag $F = k v^2$, $k = 0.004$ kg/m (illustrative). Simulate 10 s with Δt = 0.01 s. What is its speed at the end?

*Given:* $a = g - (k/m)\,v^2$ (down positive), $g = 9.81$ m/s²; the state is $v$; 1000 steps.

**Predict.** Speed at 10 s, in m/s. Write your prediction in the next cell *before* you run anything below. It is a variable, so it is saved with the notebook.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
m, k, g, dt = 0.16, 0.004, 9.81, 0.01
v, vs = 0.0, []
for n in range(1000):                 # 10 s
    a = g - k * v * v / m             # gravity down, drag up
    v = v + a * dt
    vs.append(v)
print(f"v(10 s) = {v:.2f} m/s, a(10 s) = {a:.1e} m/s^2")
t = dt * np.arange(1, 1001)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(t, vs, label="loop"); ax.axhline((m * g / k) ** 0.5, ls="--", c="gray", label="$\\sqrt{mg/k}$")
ax.set_xlabel("t (s)"); ax.set_ylabel("v (m/s)"); ax.legend(); plt.show()

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which of your assumptions was wrong.

In [ ]:
# Check A: independent method: terminal speed and the exact solution v_t tanh(g t / v_t)
vt = (m * g / k) ** 0.5
print(f"v_t = {vt:.2f} m/s; exact v(2 s) = {vt * math.tanh(g * 2 / vt):.2f} m/s, loop v(2 s) = {vs[199]:.2f} m/s")
# Check B: limiting case: the forces balance at the end, and v = g t (the trap) is far off
assert abs(a) < 0.01, "at 10 s the acceleration should be almost zero"
print(f"trap v = g t = {g * 10:.1f} m/s")

**Your turn.** Replace the drag by $F = b v$ with $b = 0.05$ kg/s (same ball). Find the terminal speed, the time constant τ = m/b and the simulated speed at t = τ. What fraction of $v_t$ is it?

*Hint:* terminal speed when mg = bv; the exact solution is $v_t(1 - e^{-t/\tau})$.

In [ ]:
# TODO: write your answer here

## Problem 2 · Where does the ball land?
A ball leaves the ground at 20 m/s and 45°. The loop steps it with Δt = 0.1 s and stops with `if y < 0: break`. Which x should you report as the range: the last point, or something else? How far off is the last point?

*Given:* the exact update for constant g: $x \mathrel{+}= v_x\Delta t$, $y \mathrel{+}= v_y\Delta t - \tfrac12 g\Delta t^2$, $v_y \mathrel{-}= g\Delta t$ (no Euler error); $R = v_0^2 \sin 2\theta / g$.

**Predict.** Last point x and the range you would report (m). Write your prediction in the next cell *before* you run anything below. It is a variable, so it is saved with the notebook.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
g, dt = 9.81, 0.1
x, y, vx, vy = 0.0, 0.0, 20 * 0.5**0.5, 20 * 0.5**0.5
xs, ys = [x], [y]
while True:
    xo, yo = x, y
    x, y, vy = x + vx*dt, y + vy*dt - g*dt*dt/2, vy - g*dt
    xs.append(x); ys.append(y)
    if y < 0:
        break
x_land = xo + (x - xo) * yo / (yo - y)          # straight line between the last two points
print(f"last point x = {x:.2f} m, interpolated x = {x_land:.2f} m")
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(xs, ys, "o-", ms=3); ax.axhline(0, c="k", lw=0.8); ax.plot(x_land, 0, "g*", ms=12)
ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)"); plt.show()

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which of your assumptions was wrong.

In [ ]:
# Check A: independent method: the range formula
R = 20**2 * math.sin(math.radians(90)) / g
print(f"R = {R:.2f} m, T = {2 * 20 * 0.5**0.5 / g:.3f} s")
assert abs(x_land - R) < 0.01, "the interpolated landing point should match the formula"
# Check B: convergence: a smaller step changes the last point, not the interpolated one
for d in (0.05, 0.01):
    x, y, vy = 0.0, 0.0, vx
    while True:
        xo, yo = x, y
        x, y, vy = x + vx*d, y + vy*d - g*d*d/2, vy - g*d
        if y < 0:
            break
    print(f"dt = {d}: last point {x:.2f} m, interpolated {xo + (x - xo) * yo / (yo - y):.2f} m")

**Your turn.** The same throw starts from the top of a 10 m cliff and lands on the beach below. Change only the start and find the flight time and the range. Check with the quadratic formula.

*Hint:* start at y = 10 m and keep the same stop test; $y(t) = 10 + v_{0y}t - \tfrac12 g t^2 = 0$.

In [ ]:
# TODO: write your answer here

## Problem 3 · A pendulum simulated for 20 s: which order keeps 0.20 rad?
A 1.0 m pendulum starts at θ₀ = 0.20 rad at rest. Simulate 20 s with Δt = 0.01 s in both orders: **A** (both updates from the old state) and **B** (new ω first, then θ with it). What maximum angle does each show in the last 2 s?

*Given:* $d\omega/dt = -(g/L)\sin\theta$, $d\theta/dt = \omega$; 2000 steps; small-angle period $2\pi\sqrt{L/g} = 2.006$ s.

**Predict.** Maximum |θ| in the last 2 s for A and for B (rad). Write your prediction in the next cell *before* you run anything below. It is a variable, so it is saved with the notebook.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
g, L, dt = 9.81, 1.0, 0.01
fig, ax = plt.subplots(figsize=(7, 3.5))
for order in ("A", "B"):
    th, om, peak, ths = 0.20, 0.0, 0.0, []
    for n in range(2000):                       # 20 s
        if order == "A": th, om = th + om*dt, om - g/L*math.sin(th)*dt
        else: om = om - g/L*math.sin(th)*dt; th = th + om*dt
        if n >= 1800: peak = max(peak, abs(th))
        ths.append(th)
    print(order, f"{peak:.3f} rad")
    ax.plot(dt * np.arange(1, 2001), ths, label=order)
ax.set_xlabel("t (s)"); ax.set_ylabel("θ (rad)"); ax.legend(); plt.show()
thB = np.array(ths)                             # order B, used in Check B

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which of your assumptions was wrong.

In [ ]:
# Check A: estimate: small-angle energy factor 1 + (g/L) dt^2 per step for order A
f = (1 + g / L * dt**2) ** 2000
print(f"energy x{f:.2f}, amplitude x{f**0.5:.2f} -> {0.2 * f**0.5:.2f} rad (estimate for A)")
# Check B: independent quantity: period of B from upward zero crossings
up = np.where((thB[:-1] < 0) & (thB[1:] >= 0))[0]
print(f"period B = {np.diff(up).mean() * dt:.3f} s vs 2 pi sqrt(L/g) = {2 * math.pi * (L / g) ** 0.5:.3f} s")

**Your turn.** With order B and Δt = 0.005 s, release the pendulum at θ₀ = 2.0 rad. Measure the period from upward zero crossings. How does it compare with $2\pi\sqrt{L/g}$?

*Hint:* start from your order-B loop and append θ to a list `th`; `c = [n for n in range(len(th)-1) if th[n] < 0 <= th[n+1]]`; period = (c[-1] − c[0])/(len(c) − 1) × Δt.

In [ ]:
# TODO: write your answer here

## Problem 4 · A version of the drop code passes the free-fall test. Is it right?
`fall(h, k, dt)` returns the landing time and impact speed of the Problem 1 ball (m = 0.16 kg, k = 0.004 kg/m) dropped from 50 m.
A colleague's version, with the drag sign flipped (`g + k*v*v/m`), passes the test `fall(50, 0)` ≈ $\sqrt{2h/g}$ = 3.19 s. Is that version right?

*Given:* $v_t$ = 19.81 m/s from Problem 1; the test case k = 0 has the exact answer $\sqrt{2h/g}$; time is counted with an integer step n.

**Predict.** Is the flipped version right? Which line did the k = 0 test never use? Write your prediction in the next cell *before* you run anything below. It is a variable, so it is saved with the notebook.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
def fall(h, k, dt=0.01, m=0.16, g=9.81):
    y, v, n = h, 0.0, 0                      # v positive downwards
    while y >= 0:
        y_old = y
        y, v, n = y - v*dt, v + (g - k*v*v/m)*dt, n + 1
    return (n - 1)*dt + dt*y_old/(y_old - y), v
print([round(x, 2) for x in fall(50, 0) + fall(50, 0.004)], round(math.sqrt(2*50/9.81), 2))

def fall_flipped(h, k, dt=0.01, m=0.16, g=9.81):   # the colleague's version: drag sign flipped
    y, v, n = h, 0.0, 0
    while y >= 0:
        y_old = y
        y, v, n = y - v*dt, v + (g + k*v*v/m)*dt, n + 1
    return (n - 1)*dt + dt*y_old/(y_old - y), v
print([round(x, 2) for x in fall_flipped(50, 0) + fall_flipped(50, 0.004)])

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which of your assumptions was wrong.

In [ ]:
# Check A: limiting case: with k = 0 both versions reproduce free fall (3.20 s vs 3.19 s)
for f in (fall, fall_flipped):
    assert abs(f(50, 0)[0] - math.sqrt(2 * 50 / 9.81)) < 0.05, f.__name__
print("k = 0 test passed by both versions")
# Check B: physical bound: the impact speed cannot exceed the terminal speed 19.81 m/s
for f in (fall, fall_flipped):
    t_land, v_imp = f(50, 0.004)
    print(f"{f.__name__:13s} t = {t_land:.2f} s, impact {v_imp:.2f} m/s, bound ok: {v_imp <= 19.81}")

**Your turn.** Is the drag result converged? Compute `fall(50, 0.004, dt)` for dt = 0.02, 0.01 and 0.005 s. How do the landing times change, and what is the ratio of successive differences? What does it tell you about the method?

*Hint:* print the three times with 4 decimals; ratio = (t₁ − t₂)/(t₂ − t₃).

In [ ]:
# TODO: write your answer here

## Problem 5 · A pendulum file with a −999 in it
`pendulum.csv` has 20 rows: lengths 0.20 to 1.00 m, four trials each, the period in s (illustrative). Compute the mean period at each length and $g = 4\pi^2 L/\langle T\rangle^2$. What does the raw mean at 0.80 m give?

*Given:* the header `length_m,trial,period_s` comes after two comment lines; a missed reading is −999; one period was typed in ms. **Print the file before you compute.**

**Predict.** Raw mean period at 0.80 m (s). Write your prediction in the next cell *before* you run anything below. It is a variable, so it is saved with the notebook.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
d = np.genfromtxt("pendulum.csv", delimiter=",", names=True, skip_header=2)
L, T = d["length_m"], d["period_s"]
print("raw mean at 0.80 m:", T[L == 0.80].mean().round(2), "s")
good = T > 0                                # drop the -999 sentinel
T = np.where(T > 100, T / 1000, T)          # a value in ms -> s
for Lv in np.unique(L):
    Tm = T[(L == Lv) & good].mean()
    print(Lv, round(Tm, 4), round(4 * math.pi**2 * Lv / Tm**2, 3))

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which of your assumptions was wrong.

In [ ]:
# Check A: count: one sentinel removed, the ms row kept after conversion
print(good.sum(), "of", good.size, "rows kept")
assert good.sum() == 19
# Check B: magnitude: every g within 1 % of 9.81, and the 1.00 m period matches 2 pi sqrt(L/g)
gs = [4 * math.pi**2 * Lv / T[(L == Lv) & good].mean()**2 for Lv in np.unique(L)]
print("mean g =", round(np.mean(gs), 3), "m/s^2;", "2 pi sqrt(1/9.81) =", round(2 * math.pi / 9.81**0.5, 3), "s")
assert all(abs(gi / 9.81 - 1) < 0.01 for gi in gs)

**Your turn.** Another group writes a missed reading as 0 instead of −999. Put a 0 in place of −999 in a copy of the period column and compute ⟨T⟩ and g at 0.80 m with and without the mask `T > 0`. Why is a 0 harder to spot?

*Hint:* `Tz = np.where(d['period_s'] == -999, 0, d['period_s'])`, then fix the ms value as above.

In [ ]:
# TODO: write your answer here

## Problem 6 · g from the same file, with an error bar
Use the cleaned pendulum data of Problem 5 (`L`, `T`, `good`). Fit $T^2$ against $L$ and get g from the slope, with its uncertainty. Is 9.81 m/s² compatible with your result?

*Given:* $T^2 = (4\pi^2/g)\,L$, a line through the origin; the standard error of a mean is $s/\sqrt{n}$; `np.polyfit(..., cov=True)`.

**Predict.** g with its error bar (m/s²). Write your prediction in the next cell *before* you run anything below. It is a variable, so it is saved with the notebook.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
Ls, Tm, se = [], [], []
for Lv in np.unique(L):
    Tk = T[(L == Lv) & good]
    Ls.append(Lv); Tm.append(Tk.mean()); se.append(Tk.std(ddof=1) / len(Tk)**0.5)
p, cov = np.polyfit(Ls, np.array(Tm)**2, 1, cov=True)      # T^2 = a L + b
g = 4 * math.pi**2 / p[0]; dg = g * cov[0, 0]**0.5 / p[0]
print(f"g = {g:.3f} ± {dg:.3f} m/s², b = {p[1]*1e3:+.1f} ms²")
print(f"std err at 0.20 m: {se[0]*1e3:.1f} ms")
Ls, Tm, se = map(np.array, (Ls, Tm, se))
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.errorbar(Ls, Tm**2, yerr=2 * Tm * se, fmt="o", capsize=3)
xx = np.linspace(0, 1.05, 50); ax.plot(xx, p[0] * xx + p[1])
ax.set_xlabel("L (m)"); ax.set_ylabel("T² (s²)"); plt.show()

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which of your assumptions was wrong.

In [ ]:
# Check A: physics: the intercept must be compatible with 0
print(f"b = {p[1]*1e3:+.1f} ± {cov[1, 1]**0.5*1e3:.1f} ms^2")
# Check B: independent value: distance of 9.81 from the fit, in sigma
print(f"9.81 is {abs(g - 9.81) / dg:.1f} sigma away; trap: std instead of std err at 0.20 m = {se[0] * 2 * 1e3:.1f} ms")

**Your turn.** The physics says b = 0. Fit $T^2 = aL$ with no intercept (least squares gives $a = \sum L T^2 / \sum L^2$) and compute g. Does it move?

*Hint:* use the arrays `Ls` and `Tm` from above; one line with `np.sum`.

In [ ]:
# TODO: write your answer here

## Problem 7 · A spectrum in channels: what is the unknown peak?
A detector spectrum (simulated) is recorded in ADC channels 0–4095. It shows Cs-137 (661.657 keV), Co-60 (1173.228 and 1332.492 keV) and one unknown peak. Calibrate channels to energy with two known lines, $E = a\cdot\mathrm{ch} + b$, and identify the unknown.

*Given:* `centroid(lo, hi)` below returns a peak position in channels; candidate lines: K-40 1460.8 keV, Bi-214 609.3 keV. Run the next cell first: it simulates the spectrum and plots it.

In [ ]:
# the spectrum (simulated): gain and offset are hidden inside; do not use them
rng = np.random.default_rng(2028)
E_lines = [661.657, 1173.228, 1332.492, 1460.82]
ev = [rng.normal(E, 0.06 * 662 / 2.3548 * math.sqrt(E / 662), n) for E, n in zip(E_lines, (6000, 3000, 3000, 3000))]
ev.append(rng.exponential(300, 5000))
counts, _ = np.histogram((np.concatenate(ev) + 60.0) / 0.75, bins=np.arange(0, 4097))

def centroid(lo, hi, half=40):
    # peak position in [lo, hi): smooth over 9 channels, take the maximum, then the count-weighted mean
    # channel within +/- half channels after subtracting a straight background through the window edges
    sm = np.convolve(counts, np.ones(9) / 9, mode="same")
    k = lo + int(np.argmax(sm[lo:hi]))
    i = np.arange(k - half, k + half + 1); c = counts[i].astype(float)
    bg = np.linspace(c[:5].mean(), c[-5:].mean(), c.size)
    return float(np.sum((i + 0.5) * (c - bg)) / np.sum(c - bg))

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(np.arange(4096), counts, lw=0.6); ax.set_xlim(0, 2600)
ax.set_xlabel("channel"); ax.set_ylabel("counts per channel"); plt.show()

**Predict.** Energy of the unknown peak (keV), and which nuclide? Write your prediction in the next cell *before* you run anything below. It is a variable, so it is saved with the notebook.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
c_cs, c_co, c_x = centroid(900, 1050), centroid(1780, 1940), centroid(1960, 2120)
a = (1332.492 - 661.657) / (c_co - c_cs)      # keV per channel
b = 661.657 - a * c_cs                         # keV at channel 0
print(round(c_cs, 1), round(c_co, 1), round(c_x, 1))
print(round(a, 4), round(b, 1), round(a * c_x + b, 1))

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which of your assumptions was wrong.

In [ ]:
# Check A: independent line: the Co-60 1173 keV peak must come out right
c_173 = centroid(1600, 1760)
print(f"1173 line: channel {c_173:.1f} -> {a * c_173 + b:.1f} keV (true 1173.2)")
# Check B: magnitude, and the trap E proportional to channel
print(f"gain {a:.4f} keV/ch, offset {b:.1f} keV; trap (no offset): unknown at {c_x * 661.657 / c_cs:.0f} keV")

**Your turn.** Radon daughters give a Bi-214 line at 609.3 keV. With your calibration, at which channel would it appear? The peak width there is about 22 channels (σ). Would it be separated from the Cs-137 peak?

*Hint:* invert the calibration: ch = (E − b)/a; compare the separation with σ.

In [ ]:
# TODO: write your answer here

## Problem 8 · A small bump above the background: is it real?
A peak window holds 1150 counts. Two side bands of twice the window's width hold 2000 counts together (illustrative). What are the net counts and how many σ is the bump?

*Given:* background in the window $B$ = side/2; $\sigma_B = \sqrt{\text{side}}/2$; $\sigma_\text{net}^2 = N_\text{window} + \sigma_B^2$. Compute by hand first.

**Predict.** Net counts, their σ, and the significance in σ. Write your prediction in the next cell *before* you run anything below. It is a variable, so it is saved with the notebook.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
N_win, N_side = 1150, 2000           # side bands: twice the window width
B = N_side / 2; sB = N_side**0.5 / 2
net, s_net = N_win - B, (N_win + sB**2)**0.5
print(net, round(s_net, 1), round(net / s_net, 1))

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which of your assumptions was wrong.

In [ ]:
# Check A: simulation: how often does background alone give net >= 150?
rng = np.random.default_rng(405)
fake = rng.poisson(1000, 200_000) - rng.poisson(2000, 200_000) / 2
print((fake >= net).mean(), round(fake.std(), 1))
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(fake, bins=80); ax.axvline(net, c="r")
ax.set_xlabel("net counts (background only)"); ax.set_ylabel("simulations"); plt.show()
# Check B: the spread of the background-only nets is sqrt(1000 + 500); the trap sqrt(net) is not
print(f"sqrt(1500) = {1500**0.5:.1f}; trap sqrt(net) = {net**0.5:.1f}")

**Your turn.** If the bump is real, all counts grow in proportion to the measuring time. How many times longer must you measure to reach 5 σ?

*Hint:* net ∝ t and σ ∝ √t, so the significance ∝ √t.

In [ ]:
# TODO: write your answer here

## Appendix · The two programs of the lecture (slide L03)
A 0.50 kg mass on a spring, k = 2.0 N/m, x = 0.10 m at rest, Δt = 0.05 s, 10 periods (628 steps). Only the order of the two update lines differs.

In [ ]:
w, dt = 2.0, 0.05                    # omega = sqrt(k/m) = 2 rad/s
x, v = 0.10, 0.0
for n in range(628):                 # A: both lines use the old state (Euler)
    x, v = x + v*dt, v - w*w*x*dt
print("A amplitude:", round(math.hypot(x, v / w), 3), "m")
x, v = 0.10, 0.0
for n in range(628):                 # B: new v first, then x with it (Euler-Cromer)
    v = v - w*w*x*dt
    x = x + v*dt
print("B amplitude:", round(math.hypot(x, v / w), 3), "m")

## Before you close the tab
- **Runtime → Restart session and run all**: every cell must run in order without errors.
- **File → Save**; check the notebook is in your Drive.
- Your eight predictions: count how many were right and bring the number to Week 2.
- Mistakes we met: v = gt with drag (P1), the last point as the landing (P2), the line order (P3), trusting one test (P4), −999 averaged (P5), std instead of std err (P6), E ∝ channel (P7), √net (P8).
- Each has a check: terminal speed, interpolation, energy, a physical bound, count of rows, intercept, a second line, simulation.